# Stage 1: Data Ingestion

## Goal

Raw CSV files are often too large for standard pandas loading on most machines. A 3GB CSV file can consume 10GB+ of RAM when loaded entirely into memory.

**Our Strategy:**
- Stream the CSV in chunks (100K rows at a time)
- Cast columns to memory-efficient dtypes during ingestion
- Persist the result to Parquet format

**Why Parquet?**
Parquet is a columnar storage format that:
- Compresses data by column, exploiting patterns in similar data types
- Uses dictionary encoding for string columns
- Reduces a 3GB CSV to under 500MB of disk storage
- Loads *only the columns you need*, not the entire file
- Supports efficient filtering at the storage layer

**What to Expect at the End:**
A `.parquet` file saved to `data/parquet/train.parquet` (~500MB) ready for Stage 2 cohort selection.

## Step 1: Import Libraries and Configure Logging

**What each library does:**
- `pandas`: DataFrames and data manipulation
- `pyarrow`: Parquet serialization engine
- `os`: File path and directory creation
- `logging`: Track progress during long operations (chunk-by-chunk ingestion)
- `warnings`: Suppress pandas future warnings to keep output clean

**Why logging matters:**
Ingestion can take 5-10 minutes. Logging shows us progress every 10 chunks so we know the script is still running.

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import os
import logging
import warnings

# Suppress pandas future warnings to keep output clean
warnings.filterwarnings('ignore')

# Configure logging to track ingestion progress
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

print('✓ Libraries loaded successfully')

## Step 2: Define File Paths

All paths are defined in one place. This makes the notebook easy to adapt:
- Change `RAW_CSV_PATH` if your input file is named differently
- Change `PARQUET_OUTPUT_PATH` if you want a different output location
- Adjust `CHUNK_SIZE` if you have more/less RAM available

The code handles running from any directory by navigating to the project root.

In [ ]:
# Navigate to project root (parent of notebooks directory)
# This works whether you run from the repo root or from anywhere else
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'notebooks':
    os.chdir('..')

# Now define paths relative to project root
RAW_CSV_PATH = 'data/raw/train_ver2.csv'
PARQUET_OUTPUT_PATH = 'data/parquet/train.parquet'
CHUNK_SIZE = 100_000

# Create output directory if it does not exist
os.makedirs(os.path.dirname(PARQUET_OUTPUT_PATH), exist_ok=True)

# Confirmation
print(f'Current working directory: {os.getcwd()}')
print(f'Raw CSV path: {RAW_CSV_PATH}')
print(f'Parquet output path: {PARQUET_OUTPUT_PATH}')
print(f'Chunk size: {CHUNK_SIZE:,} rows per chunk')

## Step 3: Define the 24 Product Columns and Schema

**The Dataset Structure:**
The Santander dataset has 48 columns:
- 24 customer feature columns (customer ID, demographics, relationship info, income, etc.)
- 24 binary product flags (0=no product, 1=has product)

**Why Enforce dtypes During Ingestion?**
- By default, pandas reads all numeric columns as `float64` (8 bytes per value)
- Product columns are binary (0 or 1), so they only need 1 byte
- Enforcing correct dtypes during ingestion prevents memory bloat
- We use `Int8` (nullable integer) for products because it safely handles missing values

**Memory Savings:**
- Default float64 for 24 products on 13M rows: ~2.5 GB
- Int8 for 24 products on 13M rows: ~312 MB
- **That's an 8x reduction just from smarter dtypes!**

In [ ]:
# 24 Product columns - binary indicators (0/1) for each Santander product
PRODUCT_COLS = [
    'ind_ahor_fin_ult1',      # Savings account
    'ind_aval_fin_ult1',      # Avalista
    'ind_cco_fin_ult1',       # Checking account
    'ind_cder_fin_ult1',      # Credit card
    'ind_cno_fin_ult1',       # Loan
    'ind_ctju_fin_ult1',      # Junior account
    'ind_ctma_fin_ult1',      # Plus account
    'ind_ctop_fin_ult1',      # Optimal account
    'ind_ctpp_fin_ult1',      # Premium account
    'ind_deco_fin_ult1',      # Debit card
    'ind_deme_fin_ult1',      # Demographic details requested
    'ind_dela_fin_ult1',      # Delegated account
    'ind_ecue_fin_ult1',      # E-account
    'ind_fond_fin_ult1',      # Mutual fund
    'ind_hip_fin_ult1',       # Home loan
    'ind_plan_fin_ult1',      # Pension plan
    'ind_pres_fin_ult1',      # Mortgage
    'ind_reca_fin_ult1',      # Payment facility
    'ind_tjcr_fin_ult1',      # Credit card
    'ind_valo_fin_ult1',      # Securities account
    'ind_viv_fin_ult1',       # Home insurance
    'ind_nomina_ult1',        # Payroll account
    'ind_nom_pens_ult1',      # Pension transfer
    'ind_recibo_ult1'         # Direct debit
]

# Customer feature columns with efficient string dtypes
FEATURE_DTYPES = {
    'ncodpers': 'int32',           # Customer ID
    'ind_empleado': 'str',          # Employee flag
    'pais_residencia': 'str',       # Country of residence
    'sexo': 'str',                  # Gender
    'age': 'str',                   # Age
    'fecha_alta': 'str',            # Date of account opening
    'ind_nuevo': 'str',             # New customer flag
    'antiguedad': 'str',            # Seniority (months as customer)
    'indrel': 'str',                # Customer relation type
    'ult_fec_cli_1t': 'str',        # Last date of primary account activity
    'indrel_1mes': 'str',           # Customer relation type (1 month ago)
    'tiprel_1mes': 'str',           # Type of relationship (1 month ago)
    'indresi': 'str',               # Residence flag
    'indext': 'str',                # Foreigner flag
    'conyuemp': 'str',              # Spouse employee flag
    'canal_entrada': 'str',         # Entry channel
    'indfall': 'str',               # Fallen customer indicator
    'tipodom': 'str',               # Address type
    'cod_prov': 'str',              # Province code
    'nomprov': 'str',               # Province name
    'ind_actividad_cliente': 'str', # Activity indicator
    'renta': 'str',                 # Income
    'segmento': 'str'               # Customer segment
}

print(f'✓ {len(PRODUCT_COLS)} product columns defined')
print(f'✓ {len(FEATURE_DTYPES)} feature columns with dtypes defined')

## Step 4: Stream CSV in Chunks and Convert to Parquet

**This is the most important cell in the notebook.**

**The Chunked Reading Strategy:**
Instead of loading 13 million rows at once (which requires 10GB+ RAM), we read 100K rows at a time:
1. Load 100K rows into memory
2. Cast columns to optimal dtypes
3. Append to a list
4. Repeat 130 times
5. Concatenate all chunks at the end

This avoids Out-of-Memory (OOM) errors and keeps memory usage stable at ~2GB.

**Skip Logic:**
If the Parquet file already exists, we skip ingestion and load directly. This saves time when re-running the notebook.

**Expected Output:**
- Shape: (13,000,000 rows, 48 columns) — approximately
- `fecha_dato` column: datetime64[ns]
- 24 product columns: Int8 (much smaller than float64)

⏱️ **This cell will take 5-10 minutes. Watch the log output to see progress every 10 chunks.**

In [ ]:
# Check if Parquet already exists to avoid re-ingesting
if os.path.exists(PARQUET_OUTPUT_PATH):
    logging.info(f'Parquet file already exists at {PARQUET_OUTPUT_PATH}, loading directly...')
    df = pd.read_parquet(PARQUET_OUTPUT_PATH, engine='pyarrow')
else:
    logging.info(f'Starting CSV ingestion from {RAW_CSV_PATH}...')
    
    # Create a reader object instead of loading the entire CSV
    # This returns an iterator that yields chunks of CHUNK_SIZE rows
    reader = pd.read_csv(
        RAW_CSV_PATH,
        chunksize=CHUNK_SIZE,
        dtype=FEATURE_DTYPES
    )
    
    chunks = []  # List to store each chunk
    chunk_num = 0
    
    # Iterate through chunks
    for chunk in reader:
        chunk_num += 1
        
        # Cast fecha_dato to datetime (more efficient than string)
        chunk['fecha_dato'] = pd.to_datetime(chunk['fecha_dato'], format='%Y-%m-%d')
        
        # Cast product columns to Int8 (binary 0/1 values, nullable)
        for col in PRODUCT_COLS:
            chunk[col] = chunk[col].astype('Int8')
        
        chunks.append(chunk)
        
        # Log progress every 10 chunks
        if chunk_num % 10 == 0:
            cumulative_rows = chunk_num * CHUNK_SIZE
            logging.info(f'Chunk {chunk_num}: {cumulative_rows:,} rows processed')
    
    # Concatenate all chunks into a single dataframe
    logging.info('Concatenating all chunks...')
    df = pd.concat(chunks, ignore_index=True)
    
    # Write to Parquet with compression
    logging.info(f'Writing to Parquet: {PARQUET_OUTPUT_PATH}')
    df.to_parquet(PARQUET_OUTPUT_PATH, engine='pyarrow', compression='snappy')
    
    logging.info(f'✓ Ingestion complete. DataFrame shape: {df.shape}')

## Step 5: Validate the Ingested Data

Before moving to Stage 2 (Cohort Selection), we must confirm the data loaded correctly.

**What healthy output looks like:**
- Shape: ~13 million rows × 48 columns
- `fecha_dato` dtype: `datetime64[ns]`
- 24 product columns dtype: `Int8` (not float64)
- No unexpected nulls or errors

If dtypes don't match expectations, we may need to adjust the ingestion logic above.

In [ ]:
# Display dataframe shape
print('Dataframe Shape:')
print(f'  {df.shape[0]:,} rows × {df.shape[1]} columns\n')

# Display all dtypes
print('Data Types:')
print(df.dtypes)
print()

# Confirm fecha_dato is datetime
print('Unique dates (sample):')
print(sorted(df['fecha_dato'].unique())[:5])
print()

# Confirm product columns are Int8
print('Product column dtypes (sample):')
print(df[PRODUCT_COLS[:5]].dtypes)
print()

# Display first 3 rows
print('First 3 rows:')
df.head(3)

## Step 6: Memory Usage Report

This cell checks how much RAM the dataframe is consuming.

**Why this matters:**
If the total memory usage exceeds your available RAM, subsequent stages (especially Stage 2 cohort filtering) become critical to reduce size. We cannot afford to create temporary copies without careful memory management.

A healthy in-memory size is 2-4 GB for the full 13M row dataset.

In [ ]:
# Calculate total memory usage
total_memory_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f'Total Memory Usage: {total_memory_mb:,.1f} MB ({total_memory_mb/1024:.2f} GB)')
print()

# Breakdown by dtype
print('Memory by Data Type:')
for dtype, count in df.dtypes.value_counts().items():
    print(f'  {dtype}: {count} columns')
print()

print('='*60)
print(f'✓ Stage 1 Complete!')
print(f'✓ Parquet file saved to: {PARQUET_OUTPUT_PATH}')
print(f'✓ Ready for Stage 2: Cohort Selection')
print('='*60)